# Glove

In [1]:
# !pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score

# Load word embeddings (can replace with your own Word2Vec/GloVe)
model = api.load('glove-wiki-gigaword-100')  # or 'word2vec-google-news-300'

[==================================================] 100.0% 128.1/128.1MB downloaded


In [3]:
def get_sentence_vector(sentence):
    words = [word for word in sentence.lower().split() if word in model]
    if not words:
        return np.zeros(model.vector_size)
    return np.mean([model[word] for word in words], axis=0)

## Segmenting similar tasks

In [4]:
tasks = [
    "clean data", "build model", "train algorithm", "remove outliers",
    "deploy model", "visualize data", "tune hyperparameters", "generate report"
]

task_vectors = [get_sentence_vector(task) for task in tasks]

# Use KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(task_vectors)

# Print grouped tasks
for i, label in enumerate(labels):
    print(f"Cluster {label}: {tasks[i]}")

Cluster 1: clean data
Cluster 0: build model
Cluster 0: train algorithm
Cluster 1: remove outliers
Cluster 0: deploy model
Cluster 1: visualize data
Cluster 2: tune hyperparameters
Cluster 1: generate report


## Checking accuracy in summary

In [5]:
def check_summary_accuracy(original_text, summary_text):
    orig_words = [word for word in original_text.lower().split() if word in model]
    summary_words = [word for word in summary_text.lower().split() if word in model]

    orig_vecs = np.array([model[word] for word in orig_words])
    summary_vecs = np.array([model[word] for word in summary_words])

    similarities = cosine_similarity(summary_vecs, orig_vecs)
    print(similarities)
    coverage = np.mean(np.max(similarities, axis=1))  # max sim for each summary word
    return coverage

original = "The quick brown fox jumps over the lazy dog and runs into the forest."
summary = "A brown fox jumps over a lazy dog."

accuracy_score = check_summary_accuracy(original, summary)
print("Summary accuracy score (cosine-based):", accuracy_score)

[[0.7759781  0.5729593  0.52831995 0.40486136 0.2300339  0.71944714
  0.7759781  0.22158211 0.43240827 0.6219409  0.45334157 0.7065192
  0.77597797]
 [0.50965744 0.40216017 1.         0.5351951  0.11926231 0.5423247
  0.50965744 0.2397607  0.4018255  0.5389959  0.2936742  0.45475176
  0.50965744]
 [0.44629264 0.27473518 0.5351951  1.         0.15478665 0.40027076
  0.44629264 0.25738817 0.41424996 0.39838508 0.37033644 0.35306576
  0.4462927 ]
 [0.22959918 0.343118   0.11926231 0.15478665 1.0000001  0.2853942
  0.22959918 0.35188648 0.3208771  0.24358284 0.39954513 0.29147178
  0.22959916]
 [0.7932201  0.4554352  0.5423246  0.4002707  0.28539413 1.
  0.7932201  0.07971919 0.34535676 0.76089466 0.54290015 0.75144744
  0.7932201 ]
 [0.77597797 0.57295936 0.5283199  0.40486145 0.23003387 0.71944726
  0.77597797 0.2215821  0.43240827 0.6219409  0.45334157 0.7065192
  0.77597797]
 [0.14044847 0.34478733 0.23976067 0.25738826 0.35188642 0.07971919
  0.14044847 1.0000002  0.47685993 0.1277998

## Text Clustering

In [11]:
documents = [
    "Dogs are wonderful pets",
    "Cats are independent animals",
    "Dogs love to play fetch",
    "Cats love to nap",
    "Football is a great sport",
    "Soccer is popular worldwide",
]
doc_vectors = [get_sentence_vector(doc) for doc in documents]
print(np.array(doc_vectors).shape)
# print(np.array(doc_vectors)[0::6])
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(doc_vectors)

for i, label in enumerate(clusters):
    print(f"Cluster {label}: {documents[i]}")


(6, 100)
Cluster 0: Dogs are wonderful pets
Cluster 0: Cats are independent animals
Cluster 0: Dogs love to play fetch
Cluster 0: Cats love to nap
Cluster 1: Football is a great sport
Cluster 1: Soccer is popular worldwide


## Semantic Search

In [14]:
corpus = [
    "How to train a neural network",
    "Ways to clean data for machine learning",
    "Data visualization techniques",
    "Best practices for model deployment",
    "NLP is amazing",
    "MY name is vinod."
]

query = "visualizing data"
query_vec = get_sentence_vector(query)

corpus_vecs = [get_sentence_vector(doc) for doc in corpus]
sims = cosine_similarity([query_vec], corpus_vecs)[0]

results = sorted(zip(corpus, sims), key=lambda x: x[1], reverse=True)
print("Semantic Search Results:")
for doc, score in results:
    print(f"{score:.3f}: {doc}")

Semantic Search Results:
0.829: Data visualization techniques
0.634: Ways to clean data for machine learning
0.515: How to train a neural network
0.438: Best practices for model deployment
0.394: NLP is amazing
0.312: MY name is vinod.
